# Epic 0 Notebook

This notebook drives the first bbox-level CBIR baseline. The workflow is intentionally narrow:

1. Build the class manifest from the final COCO and Datumaro exports.
2. Audit bbox quality by split, camera, and size bucket.
3. Inspect one bbox-derived crop with `padding_ratio = 0.0` by default.
4. Optionally ingest the selected slice into Milvus.
5. Optionally query Milvus and inspect the top-k neighbors.
6. Compute early retrieval metrics on a controlled subset.

The canonical persisted data unit is the raw bbox annotation. Crops are derived at runtime.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from tqdm.auto import tqdm

cwd = Path.cwd().resolve()
REPO_ROOT = cwd.parent if cwd.name == "mvp" else cwd
MVP_ROOT = REPO_ROOT / "mvp"
if not MVP_ROOT.exists():
    raise RuntimeError(f"Could not locate the mvp directory from {cwd}")
if str(MVP_ROOT) not in sys.path:
    sys.path.insert(0, str(MVP_ROOT))

from data_prep import (
    DEFAULT_BENCHMARK_AREA_THRESHOLD,
    build_class_manifest,
    load_manifest,
    select_manifest_records,
    summarize_records,
    write_manifest_artifacts,
)
from db import DEFAULT_HOST, DEFAULT_PORT, search as milvus_search
from extract import collection_name_for, extract_batch, load_model
from ingest import ingest_records
from visualize import render_inspect, render_search, search_hits

target_class = "Traineira"
split = "train"
idx = 1
padding_ratio = 0.0
benchmark_only = True
top_k = 10

model_name = "openclip-vit-b-32"
device = "cpu"
collection_name = collection_name_for(target_class, model_name)
host = DEFAULT_HOST
port = DEFAULT_PORT

run_ingest = False
ingest_limit = 200
run_search = False
metric_query_limit = 20


In [ ]:
manifest_records, rejected_records = build_class_manifest(
    target_class,
    benchmark_area_threshold=DEFAULT_BENCHMARK_AREA_THRESHOLD,
    repo_root=REPO_ROOT,
)
artifact_paths = write_manifest_artifacts(
    target_class,
    manifest_records,
    rejected_records,
    repo_root=REPO_ROOT,
)
manifest_path = artifact_paths["manifest_path"]
summary = summarize_records(target_class, manifest_records, rejected_records)

records = load_manifest(manifest_path)
records_df = pd.DataFrame(records)

display(Markdown(f"**Manifest path:** `{manifest_path}`"))
display(Markdown(f"**Rejected records:** `{len(rejected_records)}`"))

split_summary_df = (
    pd.DataFrame.from_dict(summary["counts_by_split"], orient="index")
    .rename_axis("split")
    .reset_index()
)
camera_summary_df = pd.DataFrame(
    sorted(summary["counts_by_camera_id"].items()),
    columns=["camera_id", "records"],
)
size_summary_df = pd.DataFrame(
    summary["counts_by_size_bucket"].items(),
    columns=["size_bucket", "records"],
)

audit_df = records_df.copy()
if benchmark_only:
    audit_df = audit_df[audit_df["is_benchmark_candidate"]]

camera_by_split_df = (
    audit_df.groupby(["split", "camera_id"]).size().rename("records").reset_index()
)
size_by_split_df = (
    audit_df.groupby(["split", "size_bucket"]).size().rename("records").reset_index()
)

display(split_summary_df)
display(camera_summary_df)
display(size_summary_df)
display(camera_by_split_df)
display(size_by_split_df)


In [ ]:
selected_records = select_manifest_records(
    records,
    split=split,
    benchmark_only=benchmark_only,
)
if not selected_records:
    raise ValueError(
        f"No records available for class={target_class!r}, split={split!r}, benchmark_only={benchmark_only!r}"
    )
if idx < 1 or idx > len(selected_records):
    raise IndexError(f"idx must be between 1 and {len(selected_records)}; got {idx}")

record = selected_records[idx - 1]
figure, report = render_inspect(record=record, padding_ratio=padding_ratio)
display(figure)
plt.close(figure)
display(
    pd.DataFrame({
        "field": [line.split(":", 1)[0] for line in report],
        "value": [line.split(":", 1)[1].strip() for line in report],
    })
)


In [ ]:
if run_ingest:
    ingest_summary = ingest_records(
        manifest_path=manifest_path,
        collection_name=collection_name,
        model_name=model_name,
        device=device,
        split=split,
        benchmark_only=benchmark_only,
        padding_ratio=padding_ratio,
        batch_size=32,
        insert_batch_size=128,
        recreate=True,
        host=host,
        port=port,
        limit=ingest_limit,
    )
    display(pd.DataFrame([ingest_summary]).T.rename(columns={0: "value"}))
else:
    display(Markdown("Set `run_ingest = True` after `docker compose up -d` to populate Milvus."))


In [ ]:
if run_search:
    hits = search_hits(
        record=record,
        collection_name=collection_name,
        manifest_records=records,
        model_name=model_name,
        device=device,
        padding_ratio=padding_ratio,
        top_k=top_k,
        host=host,
        port=port,
    )
    search_figure, _ = render_search(record=record, hits=hits, padding_ratio=padding_ratio)
    display(search_figure)
    plt.close(search_figure)
    display(
        pd.DataFrame([
            {
                "rank": rank,
                "item_id": hit["item_id"],
                "score": hit["score"],
                "split": hit["record"]["split"],
                "camera_id": hit["record"]["camera_id"],
                "size_bucket": hit["record"]["size_bucket"],
            }
            for rank, hit in enumerate(hits, start=1)
        ])
    )
else:
    display(Markdown("Set `run_search = True` after ingesting a collection into Milvus."))


In [ ]:
if run_search:
    metric_records = select_manifest_records(
        records,
        split="test",
        benchmark_only=benchmark_only,
    )[:metric_query_limit]
    model_bundle = load_model(model_name=model_name, device=device)
    precision_rows = []
    for query_record in tqdm(metric_records, desc="Metric queries"):
        extracted = extract_batch(
            model_bundle,
            [query_record],
            batch_size=1,
            device=model_bundle["device"],
            padding_ratio=padding_ratio,
        )
        hits = milvus_search(collection_name, extracted[0]["embedding"], top_k=top_k)
        row = {"item_id": query_record["item_id"]}
        for k in (5, 10):
            if top_k >= k:
                row[f"precision_at_{k}"] = (
                    sum(1 for hit in hits[:k] if hit["target_class"] == query_record["target_class"]) / k
                )
        precision_rows.append(row)

    precision_df = pd.DataFrame(precision_rows)
    display(precision_df)
    metric_columns = [column for column in precision_df.columns if column.startswith("precision_at_")]
    if metric_columns:
        display(precision_df[metric_columns].mean().to_frame(name="mean_precision"))
    if records and len({record["target_class"] for record in records}) == 1:
        display(Markdown("Class-match precision is structurally trivial for a single-class collection. A mixed-class collection is required for a discriminative retrieval benchmark."))
else:
    display(Markdown("Metrics depend on a live Milvus collection. Keep `run_search = False` until the collection exists."))
